# Notebook 01 — Why Mathematics for Biology

**Module:** 02 — Mathematics for Biology  
**Notebook:** 01 of 12  
**Prerequisites:** Module 01 complete  
**Time estimate:** 45 minutes

---
## Step 1 — Motivation

Mathematics is not the goal — understanding biology is. But certain biological
questions cannot be answered by description alone. How many bacteria will there be
in 6 hours? Will the predator population crash? At what drug dose does 50% of cells
die? These questions require a model — a precise, quantitative statement about how
a system changes over time.

This notebook answers: *which mathematical ideas actually show up in computational
biology, and why does this module cover them in this order?*

---
## Step 2 — Intuition

Two types of models:

**Descriptive (statistical):** 'Here is the pattern in the data.' → Module 03  
**Mechanistic (mathematical):** 'Here is *why* the pattern exists.' → This module

A mechanistic model lets you predict outcomes you haven't measured — a drug dose
you haven't tested, a population size at a future time, a mutation rate you can
only estimate indirectly.

---
## Step 3 — Biological Background

**Five biological questions that require mathematics:**

| Question | Biological domain | Mathematical tool |
|----------|------------------|------------------|
| How fast does a bacterial population double? | Microbiology | Exponential growth ODE |
| Will a virus spread through a population? | Epidemiology | SIR model ODEs |
| Why do predator/prey populations oscillate? | Ecology | Lotka-Volterra ODEs |
| What drug concentration kills 50% of cells? | Pharmacology | Dose-response curve fitting |
| How many possible 10-mer DNA sequences are there? | Genomics | Combinatorics |

Every question in the table is answered in this module.

---
## Step 4 — Mathematical Explanation

**What is a differential equation?**

An ordinary differential equation (ODE) is a relationship between a function and
its derivative:

$$\frac{dN}{dt} = f(N, t)$$

It says: 'the rate of change of $N$ at time $t$ depends on the current value of
$N$ (and possibly $t$).' It does NOT say what $N(t)$ is — that is what we solve for.

**Why are ODEs everywhere in biology?**  
Because biological quantities — population size, concentration, gene expression
level — change continuously over time, and the rate of change usually depends on
the current state.

---
## Step 5 — Computational Explanation

**Module 02 dependency map:**

```
NB01 (this notebook) — orientation
├── NB02 — derivatives         → needed for NB06, NB07, NB08
├── NB03 — integrals           → needed for NB06 (AUC)
├── NB04 — exponential/logistic → first ODE, first fitting
├── NB05 — difference equations → discrete vs. continuous
├── NB06 — separable ODEs      → analytical solutions
│   ├── NB07 — Euler           → numerical solution from scratch
│   └── NB08 — RK4             → better numerical solution
│       └── NB09 — Lotka-Volterra → portfolio artifact
├── NB10 — graphs/combinatorics → discrete mathematics
├── NB11 — gradient descent    → optimization
└── NB12 — mini project        → integration and assessment
```

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Biological case studies that recur throughout the module
import numpy as np
import matplotlib.pyplot as plt

# Case 1: E. coli doubling time ~20 min in ideal conditions
# N(t) = N0 * 2^(t / T_double)

T_double = 20  # minutes
N0 = 1         # starting with 1 cell
t = np.linspace(0, 180, 500)  # 3 hours
N_ecoli = N0 * 2**(t / T_double)

print("E. coli growth:")
for time in [0, 60, 120, 180]:
    n = N0 * 2**(time / T_double)
    print(f"  t={time:3d} min → {n:,.0f} cells")

In [ ]:
# Cell 6.2 — Preview: all five biological questions answered numerically
from scipy.integrate import solve_ivp

# Question 3 preview: Lotka-Volterra oscillations
def lotka_volterra(t, y, alpha, beta, gamma, delta):
    x, y_pred = y
    return [alpha*x - beta*x*y_pred,
            delta*x*y_pred - gamma*y_pred]

sol = solve_ivp(lotka_volterra, [0, 30], [10, 5],
                args=(1.0, 0.1, 1.5, 0.075), dense_output=True)
t_plot = np.linspace(0, 30, 500)
prey, predator = sol.sol(t_plot)

print(f"Lotka-Volterra: prey oscillates between {prey.min():.1f} and {prey.max():.1f}")
print(f"                predator between {predator.min():.1f} and {predator.max():.1f}")
print("\nYou will derive and explain every line of this by NB09.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Module overview: 5 biological questions, 5 plots
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import numpy as np

rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: E. coli exponential growth
axes[0].semilogy(t/60, N_ecoli, color="steelblue", lw=2)
axes[0].set_xlabel("Hours"); axes[0].set_ylabel("Cell count (log scale)")
axes[0].set_title("E. coli growth (NB04)")

# Panel 2: Logistic map chaos
r_vals = np.linspace(2.5, 4.0, 400)
x = 0.5 * np.ones_like(r_vals)
for _ in range(300): x = r_vals * x * (1 - x)
for _ in range(100):
    x = r_vals * x * (1 - x)
    axes[1].plot(r_vals, x, ',k', alpha=0.1, markersize=0.5)
axes[1].set_xlabel("r"); axes[1].set_ylabel("N*")
axes[1].set_title("Logistic map bifurcations (NB05)")

# Panel 3: Lotka-Volterra
axes[2].plot(t_plot, prey, color="steelblue", label="Prey")
axes[2].plot(t_plot, predator, color="tomato", label="Predator")
axes[2].set_xlabel("Time"); axes[2].set_ylabel("Population")
axes[2].set_title("Lotka-Volterra (NB09)")
axes[2].legend(frameon=False)

plt.suptitle("Module 02 preview — what you'll be able to model", fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. E. coli doubles every 20 minutes. Starting from a single cell, how many cells after
   exactly 8 hours? Express your answer in scientific notation. Verify with Python.
2. A tumour doubles every 90 days. If a tumour contains 10⁶ cells when detected, how
   many cells does it have 1 year later?
3. Write a one-sentence definition of a differential equation in plain English.
   No mathematical notation allowed.
4. Looking at the module dependency map in Cell 5.1: which notebook is the prerequisite
   for the most other notebooks? Why?

---
## Quiz — Active Recall

1. What is the difference between a descriptive and a mechanistic model?
2. Why is the E. coli doubling time a rate of change, not just a count?
3. Name the five biological questions this module will answer.
4. What does `solve_ivp` stand for? What does 'IVP' mean?
5. Why does a logistic growth model have a ceiling (carrying capacity K) while
   exponential growth does not?

---
## Reflection

**Date completed:** ____________________

> *[Which of the five biological questions is most relevant to your interests? Which mathematical tool looks most unfamiliar?]*

---
*Next: `02_derivatives_and_limits.ipynb`*